In [0]:
# =============================================================================
# Phase 2 — Feature Engineering : Silver → Silver Featured
# Craigslist Used Vehicle Dataset
# =============================================================================

import logging
import time
from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)
log = logging.getLogger("VehicleETL_SilverFeatured")

# =============================================================================
# CONFIG
# =============================================================================

CURRENT_YEAR = datetime.now().year

SILVER_TABLE   = "workspace.default.silver_vehicles"
FEATURED_TABLE = "workspace.default.featured_vehicles"

LUXURY_BRANDS = {
    "bmw", "mercedes-benz", "audi", "lexus", "cadillac", "lincoln",
    "porsche", "jaguar", "land rover", "infiniti", "acura", "volvo",
    "tesla", "maserati", "ferrari", "lamborghini", "genesis"
}

ALTERNATIVE_FUELS = {"electric", "hybrid", "other"}

# Features where nulls are expected due to missing source values
# These will not trigger a warning in the validation report
EXPECTED_NULL_FEATURES = {"price_per_mile", "depreciation_index"}



In [0]:
# =============================================================================
# 1. SPARK SESSION
# =============================================================================

spark = SparkSession.builder \
    .appName("VehicleMarket_SilverFeatured") \
    .getOrCreate()

pipeline_start = time.time()
log.info("="*65)
log.info("  Phase 3 — Feature Engineering : Silver → Silver Featured")
log.info("="*65)


2026-07-26 18:08:19 | INFO | =================================================================
2026-07-26 18:08:19 | INFO |   Phase 3 — Feature Engineering : Silver → Silver Featured
2026-07-26 18:08:19 | INFO | =================================================================


In [0]:
# =============================================================================
# 2. LOAD SILVER LAYER
# =============================================================================

log.info("Loading Silver layer...")
df = spark.read.table(SILVER_TABLE)
record_count_in = df.count()
columns_in      = len(df.columns)
log.info(f"  Records loaded : {record_count_in}")
log.info(f"  Columns loaded : {columns_in}")



2026-07-26 18:08:19 | INFO | Loading Silver layer...
2026-07-26 18:08:21 | INFO |   Records loaded : 244772
2026-07-26 18:08:21 | INFO |   Columns loaded : 25


In [0]:
# =============================================================================
# 3. VEHICLE AGE & AGE GROUP
# Purpose : Vehicle age enables depreciation analysis, age-based pricing,
#           and inventory health assessment.
# Dashboards : Dashboard 1 (Avg Age KPI), Dashboard 3 (Age Distribution,
#              Price vs Age), Dashboard 2 (Price Trend by Year)
# =============================================================================

log.info("Feature 1/13 — vehicle_age, age_group...")

df = df.withColumn(
    "vehicle_age",
    (F.lit(CURRENT_YEAR) - F.col("year")).cast(IntegerType())
)

df = df.withColumn(
    "age_group",
    F.when(F.col("vehicle_age") <= 2,  "Nearly New (0-2 yrs)")
     .when(F.col("vehicle_age") <= 5,  "Recent (3-5 yrs)")
     .when(F.col("vehicle_age") <= 10, "Mid-Age (6-10 yrs)")
     .when(F.col("vehicle_age") <= 15, "Older (11-15 yrs)")
     .otherwise("Classic (15+ yrs)")
)


2026-07-26 18:08:21 | INFO | Feature 1/13 — vehicle_age, age_group...


In [0]:
# =============================================================================
# 4. PRICE CATEGORY
# Purpose : Segments market into buyer-friendly price tiers for
#           detailed distribution analysis and affordability insights.
#           Use this for all business analytics and Tableau dashboards.
#           (See also price_band for lightweight streaming use.)
# Dashboards : Dashboard 2 (Price Distribution, Price Intelligence),
#              Dashboard 3 (Price vs Mileage scatter)
# =============================================================================

log.info("Feature 2/13 — price_category...")

df = df.withColumn(
    "price_category",
    F.when(F.col("price") < 5000,  "Budget (< $5K)")
     .when(F.col("price") < 15000, "Mid-Range ($5K-$15K)")
     .when(F.col("price") < 35000, "Premium ($15K-$35K)")
     .when(F.col("price") < 75000, "High-End ($35K-$75K)")
     .otherwise("Luxury ($75K+)")
)

2026-07-26 18:08:22 | INFO | Feature 2/13 — price_category...


In [0]:
# =============================================================================
# 5. MILEAGE CATEGORY
# Purpose : Classifies vehicles by usage level, enabling buyers to
#           filter by mileage tier and supporting inventory health analysis.
# Dashboards : Dashboard 3 (Mileage Distribution, Price vs Mileage),
#              Dashboard 4 (Mileage by State)
# =============================================================================

log.info("Feature 3/13 — mileage_category...")

df = df.withColumn(
    "mileage_category",
    F.when(F.col("odometer") < 20000,  "Low (< 20K)")
     .when(F.col("odometer") < 60000,  "Moderate (20K-60K)")
     .when(F.col("odometer") < 100000, "High (60K-100K)")
     .when(F.col("odometer") < 150000, "Very High (100K-150K)")
     .otherwise("Extreme (150K+)")
)

2026-07-26 18:08:22 | INFO | Feature 3/13 — mileage_category...


In [0]:
# =============================================================================
# 6. PRICE PER MILE
# Purpose : Measures cost efficiency — how much a buyer pays per mile
#           driven. Lower values indicate better value for money.
# Note    : Nulls are expected where odometer is null or zero.
#           This is valid behavior, not a data quality issue.
# Dashboards : Dashboard 3 (Vehicle value analysis, Price vs Mileage)
# =============================================================================

log.info("Feature 4/13 — price_per_mile...")

df = df.withColumn(
    "price_per_mile",
    F.when(
        F.col("odometer").isNotNull() & (F.col("odometer") > 0),
        F.round(F.col("price") / F.col("odometer"), 4)
    ).otherwise(None)
)


2026-07-26 18:08:22 | INFO | Feature 4/13 — price_per_mile...


In [0]:
# =============================================================================
# 7. POSTING DATE FEATURES
# Purpose : Enables time-series analysis, seasonal trends, and
#           weekday/monthly listing activity patterns.
# Dashboards : Dashboard 1 (Listing Trend), Dashboard 5 (Live Trend)
# =============================================================================

log.info("Feature 5/13 — posting date features...")

df = df \
    .withColumn("posting_year",         F.year(F.col("posting_date"))) \
    .withColumn("posting_month",        F.month(F.col("posting_date"))) \
    .withColumn("posting_day",          F.dayofmonth(F.col("posting_date"))) \
    .withColumn("posting_weekday",      F.dayofweek(F.col("posting_date"))) \
    .withColumn("posting_quarter",      F.quarter(F.col("posting_date"))) \
    .withColumn("posting_month_name",   F.date_format(F.col("posting_date"), "MMMM")) \
    .withColumn("posting_weekday_name", F.date_format(F.col("posting_date"), "EEEE")) \
    .withColumn("posting_hour",         F.hour(F.col("posting_date")))



2026-07-26 18:08:23 | INFO | Feature 5/13 — posting date features...


In [0]:
# =============================================================================
# 8. POSTING SEASON
# Purpose : Groups months into seasons for seasonal listing trend
#           analysis. Complements month and quarter features.
# Dashboards : Dashboard 1 (Listing Trend seasonal filter)
# Mapping   : Winter = Dec, Jan, Feb
#             Spring = Mar, Apr, May
#             Summer = Jun, Jul, Aug
#             Fall   = Sep, Oct, Nov
# =============================================================================

log.info("Feature 6/13 — posting_season...")

df = df.withColumn(
    "posting_season",
    F.when(F.col("posting_month").isin(12, 1, 2), "Winter")
     .when(F.col("posting_month").isin(3, 4, 5),  "Spring")
     .when(F.col("posting_month").isin(6, 7, 8),  "Summer")
     .when(F.col("posting_month").isin(9, 10, 11), "Fall")
     .otherwise(None)
)


2026-07-26 18:08:23 | INFO | Feature 6/13 — posting_season...


In [0]:
# =============================================================================
# 8b. WEEKEND FLAG & POSTING PERIOD
# Purpose : is_weekend enables weekend vs weekday listing behaviour analysis.
#           posting_period segments the day into time-of-day business dimensions.
# Dashboards : Dashboard 1 (Listing activity patterns), Dashboard 5 (Live filters)
# =============================================================================

log.info("Feature 14/16 — is_weekend, posting_period...")

df = df.withColumn(
    "is_weekend",
    F.when(F.col("posting_weekday_name").isin("Saturday", "Sunday"), True)
     .otherwise(False)
)

df = df.withColumn(
    "posting_period",
    F.when(F.col("posting_hour").between(0, 5),   "Night")
     .when(F.col("posting_hour").between(6, 11),  "Morning")
     .when(F.col("posting_hour").between(12, 17), "Afternoon")
     .otherwise("Evening")
)

2026-07-26 18:08:24 | INFO | Feature 14/16 — is_weekend, posting_period...


In [0]:
# =============================================================================
# 9. LUXURY BRAND FLAG
# Purpose : Identifies premium manufacturer listings to support luxury
#           segment analysis and premium pricing trends.
# Dashboards : Dashboard 2 (Top Premium Models, Price Intelligence)
# =============================================================================

log.info("Feature 7/13 — is_luxury...")

df = df.withColumn(
    "is_luxury",
    F.when(F.col("manufacturer").isin(list(LUXURY_BRANDS)), True)
     .otherwise(False)
)

2026-07-26 18:08:24 | INFO | Feature 7/13 — is_luxury...


In [0]:
# =============================================================================
# 10. ALTERNATIVE FUEL FLAG
# Purpose : Isolates EV, hybrid, and non-gasoline vehicles for
#           fuel trend and market shift analysis.
# Dashboards : Dashboard 1 (Fuel Distribution), Dashboard 2 (Price by Fuel)
# =============================================================================

log.info("Feature 8/13 — is_alternative_fuel...")

df = df.withColumn(
    "is_alternative_fuel",
    F.when(F.col("fuel").isin(list(ALTERNATIVE_FUELS)), True)
     .otherwise(False)
)

2026-07-26 18:08:25 | INFO | Feature 8/13 — is_alternative_fuel...


In [0]:
# =============================================================================
# 11. AUTOMATIC TRANSMISSION FLAG
# Purpose : Identifies vehicles with automatic transmission to support
#           transmission market analysis and executive KPI reporting.
# Dashboards : Dashboard 2 (Transmission Analysis),
#              Dashboard 3 (Automatic % KPI)
# =============================================================================

log.info("Feature 9/13 — is_automatic...")

df = df.withColumn(
    "is_automatic",
    F.when(
        F.lower(F.col("transmission")) == "automatic",
        True
    ).otherwise(False)
)

2026-07-26 18:08:25 | INFO | Feature 9/13 — is_automatic...


In [0]:
# =============================================================================
# 12. VEHICLE SEGMENT
# Purpose : Groups vehicle types into market segments for inventory
#           analysis and buyer preference insights.
# Dashboards : Dashboard 3 (Vehicle Type Analysis, Inventory treemap)
# =============================================================================

log.info("Feature 10/12 — vehicle_segment...")

df = df.withColumn(
    "vehicle_segment",
    F.when(F.col("type").isin("sedan", "coupe", "hatchback", "convertible"),
           "Passenger Car")
     .when(F.col("type").isin("suv", "offroad"),
           "SUV / Off-Road")
     .when(F.col("type").isin("truck", "pickup"),
           "Truck")
     .when(F.col("type").isin("van", "mini-van"),
           "Van / Minivan")
     .when(F.col("type").isin("wagon"),
           "Wagon")
     .when(F.col("type").isin("bus"),
           "Bus / Commercial")
     .otherwise("Other / Unknown")
)

2026-07-26 18:08:25 | INFO | Feature 10/12 — vehicle_segment...


In [0]:
# =============================================================================
# 13. CLEAN TITLE FLAG
# Purpose : Binary indicator for title quality. Supports inventory
#           health KPIs and risk-aware pricing analysis.
# Dashboards : Dashboard 3 (Title Status, Clean Title % KPI)
# =============================================================================

log.info("Feature 11/13 — clean_title...")

df = df.withColumn(
    "clean_title",
    F.when(F.col("title_status") == "clean", True).otherwise(False)
)


2026-07-26 18:08:26 | INFO | Feature 11/13 — clean_title...


In [0]:
# =============================================================================
# 14. SALVAGE FLAG
# Purpose : Isolates salvage/rebuilt title vehicles for risk analysis
#           and price impact studies.
# Dashboards : Dashboard 3 (Salvage % KPI)
# =============================================================================

log.info("Feature 12/13 — is_salvage...")

df = df.withColumn(
    "is_salvage",
    F.when(F.col("title_status").isin("salvage", "rebuilt"), True)
     .otherwise(False)
)

2026-07-26 18:08:26 | INFO | Feature 12/13 — is_salvage...


In [0]:
# =============================================================================
# 15. PRICE BAND
# Purpose : Lightweight 3-tier price classification designed for
#           real-time streaming aggregations where price_category
#           is too granular for low-latency computation.
#           price_category → detailed business analytics (Tableau)
#           price_band     → lightweight real-time aggregations (Kafka/HBase)
# Dashboards : Dashboard 5 (Live Manufacturer Ranking, Live Fuel Distribution)
# =============================================================================

log.info("Feature 13/13 — price_band...")

df = df.withColumn(
    "price_band",
    F.when(F.col("price") < 10000, "Low")
     .when(F.col("price") < 30000, "Mid")
     .otherwise("High")
)


2026-07-26 18:08:27 | INFO | Feature 13/13 — price_band...


In [0]:
# =============================================================================
# 16. DEPRECIATION INDEX
# Purpose : Measures value retained per year of age. Lower values indicate
#           faster depreciation. Strongest additional business feature for
#           resale analysis, manufacturer value retention, and pricing trends.
# Note    : Nulls expected where vehicle_age <= 0 (current-year vehicles)
#           or price is null. Nulls are handled gracefully by Spark aggregations.
# Dashboards : Dashboard 2 (Manufacturer value retention, Depreciation trends)
# =============================================================================

log.info("Feature 16/16 — depreciation_index...")

df = df.withColumn(
    "depreciation_index",
    F.when(
        F.col("vehicle_age").isNull() |
        (F.col("vehicle_age") <= 0)  |
        F.col("price").isNull(),
        None
    ).otherwise(
        F.round(F.col("price") / F.col("vehicle_age"), 2)
    )
)

2026-07-26 18:08:27 | INFO | Feature 16/16 — depreciation_index...


In [0]:
# =============================================================================
# 17. CACHE
# =============================================================================
log.info("Skipping cache (Databricks Serverless does not support DataFrame cache).")


2026-07-26 18:08:27 | INFO | Skipping cache (Databricks Serverless does not support DataFrame cache).


In [0]:
# =============================================================================
# 18. WRITE FEATURED DATASET TO DELTA TABLE
# =============================================================================

log.info(f"Writing Featured Dataset to Delta table: {FEATURED_TABLE}")
df.write.format("delta").mode("overwrite").saveAsTable(FEATURED_TABLE)
log.info("  Featured Delta table saved.")

2026-07-26 18:08:28 | INFO | Writing Featured Dataset to Delta table: workspace.default.featured_vehicles
2026-07-26 18:08:34 | INFO |   Featured Delta table saved.


In [0]:
# =============================================================================
# 19. VALIDATION REPORT
# =============================================================================

record_count_out = df.count()
columns_out      = len(df.columns)
pipeline_end     = time.time()
execution_time   = round(pipeline_end - pipeline_start, 2)

NEW_FEATURES = [
    "vehicle_age", "age_group", "price_category", "mileage_category",
    "price_per_mile", "posting_year", "posting_month", "posting_day",
    "posting_weekday", "posting_quarter", "posting_month_name",
    "posting_weekday_name", "posting_hour",         
    "posting_season", "is_weekend", "posting_period", 
    "is_luxury", "is_alternative_fuel", "is_automatic",
    "vehicle_segment", "clean_title", "is_salvage", "price_band",
    "depreciation_index"                              
]

log.info("\n" + "="*65)
log.info("     SILVER FEATURED — VALIDATION REPORT")
log.info("="*65)
log.info(f"  Execution Time          : {execution_time}s")
log.info(f"  Records In  (Silver)    : {record_count_in}")
log.info(f"  Records Out (Featured)  : {record_count_out}")
log.info(f"  Columns In              : {columns_in}")
log.info(f"  Columns Out             : {columns_out}")
log.info(f"  Features Added          : {columns_out - columns_in}")
log.info("")
log.info("  Feature Null Check:")
log.info(f"  {'Feature':<30} {'Nulls':>8} {'Null%':>8}  Status")
log.info(f"  {'-'*60}")

all_passed = True
for feat in NEW_FEATURES:
    null_count = df.filter(F.col(feat).isNull()).count()
    null_pct   = null_count / record_count_out * 100

    if feat in EXPECTED_NULL_FEATURES:
        status = "— Expected nulls (missing source values)"
    elif null_pct == 0:
        status = "✓ OK"
    else:
        status     = "⚠ UNEXPECTED — review required"
        all_passed = False

    log.info(f"  {feat:<30} {null_count:>8} {null_pct:>7.2f}%  {status}")

log.info("")
if all_passed:
    log.info("  ✓ All features passed validation.")
else:
    log.info("  ⚠ Unexpected nulls detected — review flagged features.")

log.info("")
log.info("  Sample Feature Distributions:")
df.groupBy("age_group").count().orderBy("age_group").show(truncate=False)
df.groupBy("price_category").count().orderBy(F.desc("count")).show(truncate=False)
df.groupBy("posting_season").count().orderBy(F.desc("count")).show(truncate=False)
df.groupBy("vehicle_segment").count().orderBy(F.desc("count")).show(truncate=False)
df.groupBy("price_band").count().orderBy(F.desc("count")).show(truncate=False)
df.groupBy("is_weekend").count().orderBy("is_weekend").show(truncate=False)
df.groupBy("posting_period").count().orderBy(F.desc("count")).show(truncate=False)

log.info("")
log.info("  Final Schema:")
df.printSchema()
log.info("="*65)
log.info(f"  Featured Delta Table '{FEATURED_TABLE}' created successfully.")
log.info("="*65)

2026-07-26 18:08:35 | INFO | 
2026-07-26 18:08:35 | INFO |      SILVER FEATURED — VALIDATION REPORT
2026-07-26 18:08:35 | INFO | =================================================================
2026-07-26 18:08:35 | INFO |   Execution Time          : 15.6s
2026-07-26 18:08:35 | INFO |   Records In  (Silver)    : 244772
2026-07-26 18:08:35 | INFO |   Records Out (Featured)  : 244772
2026-07-26 18:08:35 | INFO |   Columns In              : 25
2026-07-26 18:08:35 | INFO |   Columns Out             : 49
2026-07-26 18:08:35 | INFO |   Features Added          : 24
2026-07-26 18:08:35 | INFO | 
2026-07-26 18:08:35 | INFO |   Feature Null Check:
2026-07-26 18:08:35 | INFO |   Feature                           Nulls    Null%  Status
2026-07-26 18:08:35 | INFO |   ------------------------------------------------------------
2026-07-26 18:08:35 | INFO |   vehicle_age                           0    0.00%  ✓ OK
2026-07-26 18:08:35 | INFO |   age_group                             0    0.00%  ✓ OK
2

+------------------+------+
|age_group         |count |
+------------------+------+
|Classic (15+ yrs) |102605|
|Mid-Age (6-10 yrs)|61288 |
|Older (11-15 yrs) |79454 |
|Recent (3-5 yrs)  |1425  |
+------------------+------+

+--------------------+------+
|price_category      |count |
+--------------------+------+
|Mid-Range ($5K-$15K)|100634|
|Premium ($15K-$35K) |70989 |
|Budget (< $5K)      |52476 |
|High-End ($35K-$75K)|19443 |
|Luxury ($75K+)      |1230  |
+--------------------+------+

+--------------+------+
|posting_season|count |
+--------------+------+
|Spring        |244772|
+--------------+------+

+----------------+-----+
|vehicle_segment |count|
+----------------+-----+
|Other / Unknown |76706|
|Passenger Car   |71070|
|SUV / Off-Road  |47204|
|Truck           |35432|
|Van / Minivan   |8430 |
|Wagon           |5511 |
|Bus / Commercial|419  |
+----------------+-----+

+----------+------+
|price_band|count |
+----------+------+
|Low       |115394|
|Mid       |98664 |
|High  

2026-07-26 18:08:45 | INFO | 
2026-07-26 18:08:45 | INFO |   Final Schema:
2026-07-26 18:08:45 | INFO | =================================================================
2026-07-26 18:08:45 | INFO |   Featured Delta Table 'workspace.default.featured_vehicles' created successfully.
2026-07-26 18:08:45 | INFO | =================================================================


+--------------+------+
|posting_period|count |
+--------------+------+
|Afternoon     |105897|
|Evening       |95559 |
|Night         |32533 |
|Morning       |10783 |
+--------------+------+

root
 |-- manufacturer: string (nullable = true)
 |-- model: string (nullable = true)
 |-- id: string (nullable = true)
 |-- url: string (nullable = true)
 |-- region: string (nullable = true)
 |-- region_url: string (nullable = true)
 |-- price: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- condition: string (nullable = true)
 |-- cylinders: string (nullable = true)
 |-- fuel: string (nullable = true)
 |-- odometer: integer (nullable = true)
 |-- title_status: string (nullable = true)
 |-- transmission: string (nullable = true)
 |-- VIN: string (nullable = true)
 |-- drive: string (nullable = true)
 |-- size: string (nullable = true)
 |-- type: string (nullable = true)
 |-- paint_color: string (nullable = true)
 |-- image_url: string (nullable = true)
 |-- description: stri

In [0]:
from pyspark.sql import functions as F

print("=" * 70)
print("Vehicle Type → Vehicle Segment Verification")
print("=" * 70)

# Original vehicle types
print("\nOriginal 'type' values:")
df.groupBy("type") \
  .count() \
  .orderBy(F.desc("count")) \
  .show(50, truncate=False)

print("\n" + "=" * 70)

# Engineered vehicle segments
print("\nEngineered vehicle segments:")
df.groupBy("vehicle_segment") \
  .count() \
  .orderBy(F.desc("count")) \
  .show(truncate=False)

print("\n" + "=" * 70)

# Show which original types became "Other / Unknown"
print("\nVehicle types mapped to 'Other / Unknown':")

df.filter(F.col("vehicle_segment") == "Other / Unknown") \
  .groupBy("type") \
  .count() \
  .orderBy(F.desc("count")) \
  .show(50, truncate=False)

Vehicle Type → Vehicle Segment Verification

Original 'type' values:
+-----------+-----+
|type       |count|
+-----------+-----+
|unknown    |70454|
|sedan      |49606|
|suv        |46741|
|truck      |18135|
|pickup     |17297|
|coupe      |8619 |
|hatchback  |8341 |
|other      |6252 |
|wagon      |5511 |
|van        |4634 |
|convertible|4504 |
|mini-van   |3796 |
|offroad    |463  |
|bus        |419  |
+-----------+-----+



Engineered vehicle segments:
+----------------+-----+
|vehicle_segment |count|
+----------------+-----+
|Other / Unknown |76706|
|Passenger Car   |71070|
|SUV / Off-Road  |47204|
|Truck           |35432|
|Van / Minivan   |8430 |
|Wagon           |5511 |
|Bus / Commercial|419  |
+----------------+-----+



Vehicle types mapped to 'Other / Unknown':
+-------+-----+
|type   |count|
+-------+-----+
|unknown|70454|
|other  |6252 |
+-------+-----+

